# Pasketti Phonetic Track - Colab T4 Training

**Crash-proof**: チェックポイントをGoogle Driveに保存。切断されても再実行で自動resume。

**手順**: Edit > Notebook settings > T4 GPU → 上から順に全セル実行（2回目以降も同じ）

In [ ]:
# === 1. Google Drive マウント + 設定 ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')

# Drive上の作業ディレクトリ（セッション間で永続）
DRIVE_DIR = '/content/drive/MyDrive/kaggle/pasketti-phonetic'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

# 前回のチェックポイントがあるか確認
ckpt_dir = os.path.join(DRIVE_DIR, 'latest_checkpoint')
if os.path.exists(os.path.join(ckpt_dir, 'training_state.json')):
    import json
    with open(os.path.join(ckpt_dir, 'training_state.json')) as f:
        state = json.load(f)
    print(f'Resume可能: step={state["global_step"]}, epoch={state["epoch"]}, best_cer={state["best_cer"]:.4f}')
else:
    print('チェックポイントなし（新規学習）')

In [ ]:
# === 2. コード取得 + 依存関係インストール ===
%cd /content
!rm -rf drivendata-comp
!git clone --depth 1 https://github.com/yasumorishima/drivendata-comp.git
!pip install -q -r drivendata-comp/pasketti-phonetic/requirements-train.txt requests

In [ ]:
# === 3. データダウンロード（初回のみ。Drive にキャッシュ） ===
import os, shutil

DATA_DIR = '/content/data/phonetic'
DRIVE_DATA = os.path.join(DRIVE_DIR, 'data')

# Drive にデータがあればコピー（再DL不要）
if os.path.exists(os.path.join(DRIVE_DATA, 'train_phon_transcripts.jsonl')):
    print('Drive キャッシュからデータ復元...')
    os.makedirs(DATA_DIR, exist_ok=True)
    # シンボリックリンクで高速化（コピー不要）
    !ln -sf {DRIVE_DATA}/* {DATA_DIR}/
    !ls -lh {DATA_DIR}/ | head -5
else:
    print('GitHub Artifact からダウンロード...')
    !python drivendata-comp/scripts/colab_data_download.py \
        --artifact drivendata-phonetic-data \
        --output {DATA_DIR}
    # Drive にもキャッシュ保存
    os.makedirs(DRIVE_DATA, exist_ok=True)
    !cp -r {DATA_DIR}/* {DRIVE_DATA}/
    import os as _os; _os.sync()
    print('データを Drive にキャッシュ保存完了')

In [ ]:
# === 5. 学習開始 ===
# Colab T4で1セッション完走設計:
#   batch=16, grad_accum=4 → effective=64
#   epochs=5（T4で約30〜60分で完走）
#   eval_steps=200（全168step/epoch × 5ep = 840step → 4回eval）
#
# 時間見積もり:
#   10,800 samples / batch 16 = 675 batches/epoch
#   5 epochs = 3,375 forward passes → ~30-60min on T4

import os
DRIVE_DIR = '/content/drive/MyDrive/kaggle/pasketti-phonetic'
DATA_DIR = '/content/data/phonetic'
MEMO = 'colab-v5: wav2vec2-base 5ep single-session'

!python drivendata-comp/pasketti-phonetic/train.py \
    --data_dir {DATA_DIR} \
    --output_dir /content/model_phonetic \
    --model_name facebook/wav2vec2-base \
    --epochs 5 \
    --batch_size 16 \
    --gradient_accumulation 4 \
    --lr 5e-5 \
    --warmup_steps 100 \
    --eval_steps 200 \
    --save_every_steps 0 \
    --drive_checkpoint_dir {DRIVE_DIR} \
    --memo "$MEMO"

In [ ]:
# === 5. 学習開始（resume対応） ===
# Colab T4最適設定:
#   batch_size=8, grad_accum=8 → effective=64
#   epochs=10（20だとT4 12h枠を超えるリスク）
#   save_every_steps=200（切断に備え頻繁に保存）
#   eval_steps=500

import os
DRIVE_DIR = '/content/drive/MyDrive/kaggle/pasketti-phonetic'
DATA_DIR = '/content/data/phonetic'

# resume_from: Driveにチェックポイントがあれば自動resume
RESUME = os.path.join(DRIVE_DIR, 'latest_checkpoint')
resume_flag = f'--resume_from {RESUME}' if os.path.exists(os.path.join(RESUME, 'training_state.json')) else ''

MEMO = 'colab-v5: wav2vec2-base 10ep resume'

!python drivendata-comp/pasketti-phonetic/train.py \
    --data_dir {DATA_DIR} \
    --output_dir /content/model_phonetic \
    --model_name facebook/wav2vec2-base \
    --epochs 10 \
    --batch_size 8 \
    --gradient_accumulation 8 \
    --lr 5e-5 \
    --warmup_steps 300 \
    --eval_steps 500 \
    --save_every_steps 200 \
    --drive_checkpoint_dir {DRIVE_DIR} \
    {resume_flag} \
    --memo "{MEMO}"

In [ ]:
# === 6. 結果確認 ===
!ls -lh /content/model_phonetic/final_model/
!du -sh /content/model_phonetic/final_model/

# Drive側も確認
import os
DRIVE_DIR = '/content/drive/MyDrive/kaggle/pasketti-phonetic'
drive_final = os.path.join(DRIVE_DIR, 'final_model')
if os.path.exists(drive_final):
    !du -sh {drive_final}
    print('Drive にも final_model 保存済み')
else:
    print('WARNING: Drive に final_model がありません')

In [ ]:
# === 7. GitHub Release にアップロード ===
import os
DRIVE_DIR = '/content/drive/MyDrive/kaggle/pasketti-phonetic'

# final_model をtar.gzに圧縮
MODEL_DIR = os.path.join(DRIVE_DIR, 'final_model')
if not os.path.exists(MODEL_DIR):
    MODEL_DIR = '/content/model_phonetic/final_model'

!tar czf /content/model_phonetic.tar.gz -C {MODEL_DIR} .
!ls -lh /content/model_phonetic.tar.gz

RELEASE_TAG = 'phonetic-model-v5'

!echo $GH_TOKEN | gh auth login --with-token 2>/dev/null
!gh release delete {RELEASE_TAG} --repo yasumorishima/drivendata-comp --yes 2>/dev/null; true
!gh release create {RELEASE_TAG} /content/model_phonetic.tar.gz \
    --repo yasumorishima/drivendata-comp \
    --title "{RELEASE_TAG}" \
    --notes "wav2vec2-base CTC fine-tune (Colab T4, 10 epochs)"

## 完了後: Submission パッケージ作成

```bash
gh workflow run "Package DrivenData Submission" \
  --repo yasumorishima/drivendata-comp \
  -f competition_dir=pasketti-phonetic \
  -f model_release_tag=phonetic-model-v5 \
  -f memo="colab-v5 baseline submission"
```